# Ingestão da camada bronze

A ingestão dos arquivos será feita para o catálogo de dados do Databricks na camada bronze, ainda sem nenhuma transformação.

## Configurações iniciais

- Criação do schema bronze para receber os dados brutos

In [0]:
CREATE SCHEMA IF NOT EXISTS dataquality_challenge.bronze;

In [0]:
%python
import os

raw_path = "/Workspace/Users/bueno1982@gmail.com/data-quality-challenge/raw_data"

# Lista todos os arquivos CSV na pasta de dados brutos
csv_files = [f for f in os.listdir(raw_path) if f.endswith(".csv")]

print(f"Arquivos CSV encontrados: {len(csv_files)}")
for f in csv_files:
    print(f"  - {f}")

# Ingestão de cada arquivo CSV para a camada bronze
for csv_file in csv_files:
    table_name = csv_file.replace(".csv", "")
    full_table_name = f"dataquality_challenge.bronze.{table_name}"
    file_path = f"{raw_path}/{csv_file}"

    print(f"\n--- Ingestão de {csv_file} -> {full_table_name} ---")

    try:
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(file_path)
        )

        row_count = df.count()
        print(f"  Registros lidos: {row_count:,}")
        df.printSchema()

        (
            df.write.mode("overwrite")
            .saveAsTable(full_table_name)
        )
        print(f"  Tabela {full_table_name} criada com sucesso!")

    except Exception as e:
        print(f"  Erro ao criar tabela {full_table_name}: {e}")

Arquivos CSV encontrados: 8
  - coverage_provider.csv
  - customer_territory_history.csv
  - dim_calendario.csv
  - dim_loja.csv
  - dim_produto.csv
  - fact_market_share_provider_a.csv
  - fact_market_share_provider_b.csv
  - sensitive_store_contacts.csv

--- Ingestão de coverage_provider.csv -> dataquality_challenge.bronze.coverage_provider ---
  Registros lidos: 6,370
root
 |-- provider: string (nullable = true)
 |-- year_week: string (nullable = true)
 |-- retailer_name: string (nullable = true)
 |-- expected_stores: integer (nullable = true)
 |-- received_stores: integer (nullable = true)
 |-- file_received: string (nullable = true)
 |-- expected_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)

  Tabela dataquality_challenge.bronze.coverage_provider criada com sucesso!

--- Ingestão de customer_territory_history.csv -> dataquality_challenge.bronze.customer_territory_history ---
  Registros lidos: 4,736
root
 |-- store_id: string (nullable = 

In [0]:
SELECT * FROM dataquality_challenge.bronze.coverage_provider;

provider,year_week,retailer_name,expected_stores,received_stores,file_received,expected_delivery_date,actual_delivery_date
A,2025-02,REDE 01,62,0,Y,2025-01-09,2025-01-08
A,2025-02,REDE 02,32,20,Y,2025-01-09,2025-01-11
A,2025-02,REDE 03,41,28,Y,2025-01-09,2025-01-11
A,2025-02,REDE 04,80,82,Y,2025-01-09,2025-01-09
A,2025-02,REDE 05,68,56,Y,2025-01-09,2025-01-11
A,2025-02,REDE 06,83,78,Y,2025-01-09,2025-01-08
A,2025-02,REDE 07,42,39,Y,2025-01-09,2025-01-08
A,2025-02,REDE 08,50,49,Y,2025-01-09,2025-01-09
A,2025-02,REDE 09,35,22,Y,2025-01-09,2025-01-08
A,2025-02,REDE 10,69,0,Y,2025-01-09,2025-01-08


In [0]:
SELECT * FROM dataquality_challenge.bronze.customer_territory_history;

store_id,category,territory_id,seller_id,valid_from,valid_to,assignment_source
S00001,LACTEOS,T002,V0002,2025-01-01,9999-12-31,CRM
S00001,CULINARIOS,T002,V0002,2025-01-01,9999-12-31,CRM
S00002,CHOCOLATES,T003,V0003,2025-01-01,9999-12-31,CRM
S00002,LACTEOS,T003,V0003,2025-01-01,9999-12-31,CRM
S00003,BEBIDAS,T004,V0004,2025-01-01,9999-12-31,CRM
S00003,NUTRICAO,T004,V0004,2025-01-01,9999-12-31,CRM
S00004,CHOCOLATES,T005,V0005,2025-01-01,9999-12-31,CRM
S00004,NUTRICAO,T005,V0005,2025-01-01,9999-12-31,CRM
S00005,CULINARIOS,T006,V0006,2025-01-01,9999-12-31,CRM
S00005,NUTRICAO,T006,V0006,2025-01-01,9999-12-31,CRM


In [0]:
SELECT * FROM dataquality_challenge.bronze.dim_calendario;

date,year,month,week,year_week,week_start_date,week_end_date
2025-01-06,2025,1,2,2025-02,2025-01-06,2025-01-12
2025-01-13,2025,1,3,2025-03,2025-01-13,2025-01-19
2025-01-20,2025,1,4,2025-04,2025-01-20,2025-01-26
2025-01-27,2025,1,5,2025-05,2025-01-27,2025-02-02
2025-02-03,2025,2,6,2025-06,2025-02-03,2025-02-09
2025-02-10,2025,2,7,2025-07,2025-02-10,2025-02-16
2025-02-17,2025,2,8,2025-08,2025-02-17,2025-02-23
2025-02-24,2025,2,9,2025-09,2025-02-24,2025-03-02
2025-03-03,2025,3,10,2025-10,2025-03-03,2025-03-09
2025-03-10,2025,3,11,2025-11,2025-03-10,2025-03-16


In [0]:
SELECT * FROM dataquality_challenge.bronze.dim_loja;

store_id,store_name,cnpj,retailer_name,channel,city,state,territory_id,seller_id,latitude,longitude,active_flag
S00001,LOJA 1,10000000000001,REDE 02,ATACAREJO,BELO HORIZONTE,MG,T002,V0002,-18.373227,-41.893122,Y
S00002,LOJA 2,10000000000002,REDE 03,SUPERMERCADO,UBERLANDIA,MG,T003,V0003,-26.253061,-49.107677,Y
S00003,LOJA 3,10000000000003,REDE 04,ATACAREJO,CAXIAS DO SUL,RS,T004,V0004,-28.769662,-45.004837,Y
S00004,LOJA 4,10000000000004,REDE 05,CONVENIENCIA,NITEROI,RJ,T005,V0005,-26.815782,-52.463967,Y
S00005,LOJA 5,10000000000005,REDE 06,ATACAREJO,RIO DE JANEIRO,RJ,T006,V0006,-27.269335,-40.694322,Y
S00006,LOJA 6,10000000000006,REDE 07,CONVENIENCIA,SAO PAULO,SP,T007,V0007,-26.361075,-41.641038,Y
S00007,LOJA 7,10000000000007,REDE 08,CONVENIENCIA,BELO HORIZONTE,MG,T008,V0008,-18.848966,-49.335011,Y
S00008,LOJA 8,10000000000008,REDE 09,SUPERMERCADO,LONDRINA,PR,T009,V0009,-23.180191,-45.697724,Y
S00009,LOJA 9,10000000000009,REDE 10,ATACAREJO,UBERLANDIA,MG,T010,V0010,-18.456676,-47.953878,Y
S00010,LOJA 10,10000000000010,REDE 11,ATACAREJO,PORTO ALEGRE,RS,T011,V0011,-26.758123,-48.570951,Y


In [0]:
SELECT * FROM dataquality_challenge.bronze.dim_produto;

product_id,ean,product_description,brand,manufacturer,category,subcategory,package_size,package_unit,is_own_product,valid_from,valid_to,source_system
P00001,7891000000001,NESQUIK CEREAIS 500G,NESQUIK,NESTLE,NUTRICAO,CEREAIS,500,G,Y,2024-07-23,9999-12-31,MASTER
P00002,7891000000002,NINHO CAFE 250ML,NINHO,NESTLE,BEBIDAS,CAFE,250,ML,Y,2024-09-24,9999-12-31,MASTER
P00003,7891000000003,NESCAFE CAFE 100G,NESCAFE,NESTLE,BEBIDAS,CAFE,100,G,Y,2025-01-01,9999-12-31,MASTER
P00004,7891000000004,NESQUIK CEREAIS 500G,NESQUIK,NESTLE,NUTRICAO,CEREAIS,500,G,Y,2025-01-12,9999-12-31,MASTER
P00005,7891000000005,KITKAT TEMPEROS 250ML,KITKAT,NESTLE,CULINARIOS,TEMPEROS,250,ML,Y,2024-12-27,9999-12-31,MASTER
P00006,7891000000006,GAROTO TEMPEROS 200G,GAROTO,NESTLE,CULINARIOS,TEMPEROS,200,G,Y,2025-03-28,9999-12-31,MASTER
P00007,7891000000007,nescafe leite em po 500g,NESCAFE,NESTLE,LACTEOS,LEITE EM PO,500,G,Y,2024-07-26,9999-12-31,MASTER
P00008,7891000000008,NESQUIK TABLETES 1KG,NESQUIK,NESTLE,CHOCOLATES,TABLETES,1,KG,Y,2024-05-20,9999-12-31,MASTER
P00009,7891000000009,NESQUIK LEITE EM PO 100G,NESQUIK,NESTLE,LACTEOS,LEITE EM PO,100,G,Y,2025-01-29,9999-12-31,MASTER
P00010,7891000000010,NINHO LEITE EM PO 500G,NINHO,NESTLE,LACTEOS,LEITE EM PO,500,G,Y,2024-06-08,9999-12-31,MASTER


In [0]:
SELECT COUNT(*) FROM dataquality_challenge.bronze.fact_market_share_provider_a;

COUNT(*)
68526


In [0]:
SELECT COUNT(*) FROM dataquality_challenge.bronze.fact_market_share_provider_b;

COUNT(*)
58526


In [0]:
SELECT * FROM dataquality_challenge.bronze.sensitive_store_contacts;

store_id,contact_name,contact_email,contact_phone,retailer_name
S01887,CONTATO SINTETICO 1,contato1@example.test,+55 11 910000001,REDE 33
S01440,CONTATO SINTETICO 2,contato2@example.test,+55 11 910000002,REDE 06
S00921,CONTATO SINTETICO 3,contato3@example.test,+55 11 910000003,REDE 12
S00811,CONTATO SINTETICO 4,contato4@example.test,+55 11 910000004,REDE 07
S02377,CONTATO SINTETICO 5,contato5@example.test,+55 11 910000005,REDE 33
S00203,CONTATO SINTETICO 6,contato6@example.test,+55 11 910000006,REDE 29
S00923,CONTATO SINTETICO 7,contato7@example.test,+55 11 910000007,REDE 14
S02191,CONTATO SINTETICO 8,contato8@example.test,+55 11 910000008,REDE 22
S02157,CONTATO SINTETICO 9,contato9@example.test,+55 11 910000009,REDE 23
S02266,CONTATO SINTETICO 10,contato10@example.test,+55 11 910000010,REDE 27
